# palatenet: first rating predictor

can a one-layer mean-aggregation model beat a constant training-average rating?
use boise restaurant reviews, a chronological split, and cpu training.
this is separate from the initial milestone notebook.

the graph stays frozen at the end of the earliest period. target reviews never
enter it in either direction. results apply to users and restaurants already
present in that history; coverage is reported rather than hiding the exclusions.

In [1]:
import copy
import json
import math
import platform
import statistics
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
import torch_geometric
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv

# works from the project root or this milestone folder
project_dir = Path.cwd()
if not (project_dir / "data/yelp_academic_dataset_business.json").is_file():
    project_dir = project_dir.parent
data_dir = project_dir / "data"
business_path = data_dir / "yelp_academic_dataset_business.json"
review_path = data_dir / "yelp_academic_dataset_review.json"
city, state = "Boise", "ID"
seeds = [42, 43, 44]
embedding_dim = 16
hidden_dim = 32
learning_rate = 0.005
weight_decay = 0.001
max_epochs = 150
patience = 20
device = torch.device("cpu")
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
experiment_start = time.perf_counter()

print("python:", platform.python_version())
print("torch:", torch.__version__, "pyg:", torch_geometric.__version__)
print("device:", device, "threads:", torch.get_num_threads())
print("city:", city, state, "seeds:", seeds)

python: 3.11.5
torch: 2.15.0 pyg: 2.8.1
device: cpu threads: 4
city: Boise ID seeds: [42, 43, 44]


/Users/tenzintsultrim/Class/CS224W/palatenet/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/tenzintsultrim/Class/CS224W/palatenet/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## collect the reviews

choose the city before looking at prediction results. scan the full review file,
keeping only ids, stars, and dates. no review text is retained.
for repeat user–restaurant pairs, keep the earliest valid review (file order
breaks ties). the subset isn't meant to represent all yelp users.

In [2]:
def rows(path):
    with path.open(encoding="utf-8") as source:
        for line_num, line in enumerate(source, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"bad json in {path}, line {line_num}") from error
            if not isinstance(row, dict):
                raise ValueError(f"expected an object in {path}, line {line_num}")
            yield row


def valid_id(value):
    return isinstance(value, str) and bool(value.strip())


restaurant_ids = set()
for business in rows(business_path):
    categories = business.get("categories")
    if business.get("city") != city or business.get("state") != state:
        continue
    if not isinstance(categories, str) or not valid_id(business.get("business_id")):
        continue
    if "Restaurants" in {category.strip() for category in categories.split(",")}:
        restaurant_ids.add(business["business_id"])
assert restaurant_ids

earliest = {}
matching_valid_rows = 0
invalid_city_rows = 0
scanned_rows = 0
load_start = time.perf_counter()
for review in rows(review_path):
    scanned_rows += 1
    restaurant_id = review.get("business_id")
    if not valid_id(restaurant_id) or restaurant_id not in restaurant_ids:
        continue
    user_id, stars = review.get("user_id"), review.get("stars")
    date = review.get("date")
    if (not valid_id(user_id) or isinstance(stars, bool)
            or not isinstance(stars, (int, float)) or stars not in (1, 2, 3, 4, 5)):
        invalid_city_rows += 1
        continue
    try:
        datetime.strptime(date, "%Y-%m-%d %H:%M:%S")
    except (TypeError, ValueError):
        invalid_city_rows += 1
        continue
    matching_valid_rows += 1
    pair = (user_id, restaurant_id)
    if pair not in earliest or date < earliest[pair][3]:
        earliest[pair] = (user_id, restaurant_id, float(stars), date)

# sorting the dictionary values keeps file order for timestamp ties
reviews = sorted(earliest.values(), key=lambda row: row[3])
del earliest
assert len(reviews) > 1000
assert len({(u, r) for u, r, _, _ in reviews}) == len(reviews)
assert all(stars in (1, 2, 3, 4, 5) for _, _, stars, _ in reviews)
user_counts = Counter(row[0] for row in reviews)
restaurant_counts = Counter(row[1] for row in reviews)
load_seconds = time.perf_counter() - load_start
sample_summary = {
    "city": city, "state": state, "scanned_review_rows": scanned_rows,
    "restaurant_businesses": len(restaurant_ids),
    "valid_matching_rows": matching_valid_rows, "invalid_matching_rows": invalid_city_rows,
    "duplicate_pairs_removed": matching_valid_rows - len(reviews),
    "reviews": len(reviews), "users": len(user_counts),
    "restaurants": len(restaurant_counts),
    "users_with_multiple_reviews": sum(count > 1 for count in user_counts.values()),
    "date_start": reviews[0][3], "date_end": reviews[-1][3],
    "load_seconds": load_seconds,
}
print(json.dumps(sample_summary, indent=2))

{
  "city": "Boise",
  "state": "ID",
  "scanned_review_rows": 6990280,
  "restaurant_businesses": 847,
  "valid_matching_rows": 66622,
  "invalid_matching_rows": 0,
  "duplicate_pairs_removed": 2252,
  "reviews": 64370,
  "users": 29056,
  "restaurants": 847,
  "users_with_multiple_reviews": 9582,
  "date_start": "2005-02-16 03:23:22",
  "date_end": "2022-01-19 19:31:50",
  "load_seconds": 20.30608679202851
}


## split by time

aim for 60% history, 20% training targets, 10% validation, and 10% test.
move each boundary to the next timestamp so equal timestamps stay together.
build the node maps from history alone, then filter each target period to known
users and restaurants. don't select history users based on future activity.

In [3]:
def time_boundary(fraction):
    index = int(len(reviews) * fraction)
    while index < len(reviews) and reviews[index][3] == reviews[index - 1][3]:
        index += 1
    return index


cut1, cut2, cut3 = [time_boundary(fraction) for fraction in (0.6, 0.8, 0.9)]
history = reviews[:cut1]
periods = {"train": reviews[cut1:cut2], "validation": reviews[cut2:cut3],
           "test": reviews[cut3:]}
ordered_periods = [history, periods["train"], periods["validation"], periods["test"]]
assert all(ordered_periods)
for earlier, later in zip(ordered_periods, ordered_periods[1:]):
    assert earlier[-1][3] < later[0][3]

user_map, restaurant_map = {}, {}
for user, restaurant, _, _ in history:
    if user not in user_map:
        user_map[user] = len(user_map)
    if restaurant not in restaurant_map:
        restaurant_map[restaurant] = len(restaurant_map)

targets, split_summary = {}, {}
for name, period in periods.items():
    known = [(u, r, stars, date) for u, r, stars, date in period
             if u in user_map and r in restaurant_map]
    assert known, f"no usable targets in {name}"
    targets[name] = known
    split_summary[name] = {
        "date_start": period[0][3], "date_end": period[-1][3],
        "all_pairs": len(period), "known_pairs": len(known),
        "excluded_pairs": len(period) - len(known),
        "coverage": len(known) / len(period),
        "unknown_user_pairs": sum(u not in user_map for u, _, _, _ in period),
        "unknown_restaurant_pairs": sum(r not in restaurant_map for _, r, _, _ in period),
    }
split_summary["history"] = {
    "date_start": history[0][3], "date_end": history[-1][3],
    "pairs": len(history), "users": len(user_map), "restaurants": len(restaurant_map),
}
print(json.dumps(split_summary, indent=2))

{
  "train": {
    "date_start": "2018-07-15 02:31:50",
    "date_end": "2019-12-28 03:35:51",
    "all_pairs": 12874,
    "known_pairs": 3674,
    "excluded_pairs": 9200,
    "coverage": 0.2853813888457356,
    "unknown_user_pairs": 8715,
    "unknown_restaurant_pairs": 1170
  },
  "validation": {
    "date_start": "2019-12-28 04:06:01",
    "date_end": "2021-01-24 00:52:38",
    "all_pairs": 6437,
    "known_pairs": 1043,
    "excluded_pairs": 5394,
    "coverage": 0.16203200248562996,
    "unknown_user_pairs": 5008,
    "unknown_restaurant_pairs": 1307
  },
  "test": {
    "date_start": "2021-01-24 00:58:00",
    "date_end": "2022-01-19 19:31:50",
    "all_pairs": 6437,
    "known_pairs": 794,
    "excluded_pairs": 5643,
    "coverage": 0.12334938636010565,
    "unknown_user_pairs": 5309,
    "unknown_restaurant_pairs": 1483
  },
  "history": {
    "date_start": "2005-02-16 03:23:22",
    "date_end": "2018-07-15 02:30:14",
    "pairs": 38622,
    "users": 16933,
    "restaurants": 7

## freeze the history graph

history stars are stored for inspection but never passed to the model.
only training-target stars are used as supervision, including for the baseline.
every target pair stays outside the graph, including its reverse edge.

In [4]:
def tensors(records):
    indices = torch.tensor([[user_map[u] for u, _, _, _ in records],
                            [restaurant_map[r] for _, r, _, _ in records]],
                           dtype=torch.long, device=device)
    labels = torch.tensor([stars for _, _, stars, _ in records],
                          dtype=torch.float32, device=device)
    return indices, labels


history_edges, history_stars = tensors(history)
reverse_edges = history_edges.flip(0).contiguous()
target_tensors = {name: tensors(records) for name, records in targets.items()}
graph = HeteroData()
graph["user"].num_nodes = len(user_map)
graph["restaurant"].num_nodes = len(restaurant_map)
forward = ("user", "reviews", "restaurant")
reverse = ("restaurant", "rev_reviews", "user")
graph[forward].edge_index = history_edges
graph[forward].edge_attr = history_stars[:, None]
graph[reverse].edge_index = reverse_edges
assert graph.validate(raise_on_error=True)
assert torch.equal(reverse_edges, history_edges.flip(0))

history_pairs = set(map(tuple, history_edges.t().tolist()))
reverse_pairs = set(map(tuple, reverse_edges.t().tolist()))
split_pairs = {}
for name, (indices, labels) in target_tensors.items():
    pairs = set(map(tuple, indices.t().tolist()))
    assert len(pairs) == labels.numel()
    assert pairs.isdisjoint(history_pairs)
    assert {(r, u) for u, r in pairs}.isdisjoint(reverse_pairs)
    assert indices.shape == (2, len(targets[name]))
    assert 0 <= indices[0].min() <= indices[0].max() < len(user_map)
    assert 0 <= indices[1].min() <= indices[1].max() < len(restaurant_map)
    assert torch.isin(labels, torch.arange(1, 6, device=device)).all()
    split_pairs[name] = pairs
assert split_pairs["train"].isdisjoint(split_pairs["validation"])
assert split_pairs["train"].isdisjoint(split_pairs["test"])
assert split_pairs["validation"].isdisjoint(split_pairs["test"])
original_history_edges = history_edges.clone()
print(graph)
print("time ordering, distinct pairs, indices, and both edge directions checked")

HeteroData(
  user={ num_nodes=16933 },
  restaurant={ num_nodes=740 },
  (user, reviews, restaurant)={
    edge_index=[2, 38622],
    edge_attr=[38622, 1],
  },
  (restaurant, rev_reviews, user)={ edge_index=[2, 38622] }
)
time ordering, distinct pairs, indices, and both edge directions checked


## the baseline and model

baseline: predict the mean training-target rating for every pair.

model: trainable 16-dimensional embeddings, one simultaneous mean-aggregation
layer with 32 outputs per node type, then a small network over the paired vectors.
a sigmoid maps predictions to 1–5 stars. the last layer starts at zero so initial
predictions equal the baseline. the graph stays fixed; its stored star ratings aren't used.

In [5]:
def metrics(predictions, labels):
    assert predictions.shape == labels.shape
    assert torch.isfinite(predictions).all() and torch.isfinite(labels).all()
    error = (predictions - labels).double()
    return {"mae": error.abs().mean().item(),
            "rmse": error.square().mean().sqrt().item()}


train_indices, train_labels = target_tensors["train"]
val_indices, val_labels = target_tensors["validation"]
training_mean = train_labels.mean().item()
baseline_validation = metrics(torch.full_like(val_labels, training_mean), val_labels)


class RatingModel(nn.Module):
    def __init__(self, num_users, num_restaurants, mean_rating):
        super().__init__()
        self.users = nn.Embedding(num_users, embedding_dim)
        self.restaurants = nn.Embedding(num_restaurants, embedding_dim)
        nn.init.normal_(self.users.weight, std=0.1)
        nn.init.normal_(self.restaurants.weight, std=0.1)
        self.to_users = SAGEConv((embedding_dim, embedding_dim), hidden_dim, aggr="mean")
        self.to_restaurants = SAGEConv((embedding_dim, embedding_dim), hidden_dim, aggr="mean")
        self.head = nn.Sequential(nn.Linear(2 * hidden_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, 1))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)
        probability = (mean_rating - 1) / 4
        self.register_buffer("offset", torch.tensor(math.log(probability / (1 - probability))))

    def forward(self, edges, reverse_edges, pairs):
        # both directions read the original embeddings
        user_x, restaurant_x = self.users.weight, self.restaurants.weight
        user_out = self.to_users((restaurant_x, user_x), reverse_edges).relu()
        restaurant_out = self.to_restaurants((user_x, restaurant_x), edges).relu()
        paired = torch.cat([user_out[pairs[0]], restaurant_out[pairs[1]]], dim=1)
        return 1 + 4 * torch.sigmoid(self.head(paired).squeeze(1) + self.offset)


print("training mean:", round(training_mean, 6))
print("baseline validation:", baseline_validation)

training mean: 3.777627
baseline validation: {'mae': 1.2827855322299273, 'rmse': 1.48823002033114}


## quick training check

before the real runs, try to memorize 16 training ratings on a tiny graph.
this checks gradients and the optimizer, not generalization. no validation
or test labels are used here.

In [6]:
torch.manual_seed(123)
tiny_pairs = torch.arange(16, device=device).repeat(2, 1)
tiny_labels = train_labels[:16]
tiny_edges = tiny_pairs.clone()
tiny_model = RatingModel(16, 16, training_mean).to(device)
tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.02)
initial_tiny_loss = None
for step in range(300):
    tiny_optimizer.zero_grad()
    tiny_predictions = tiny_model(tiny_edges, tiny_edges.flip(0), tiny_pairs)
    tiny_loss = nn.functional.mse_loss(tiny_predictions, tiny_labels)
    if initial_tiny_loss is None:
        initial_tiny_loss = tiny_loss.item()
    tiny_loss.backward()
    assert all(torch.isfinite(p.grad).all() for p in tiny_model.parameters() if p.grad is not None)
    tiny_optimizer.step()
assert tiny_loss.item() < 0.02, "tiny training check didn't converge"
print(f"tiny training mse: {initial_tiny_loss:.6f} -> {tiny_loss.item():.6f}")
tiny_check = {"initial_mse": initial_tiny_loss, "final_mse": tiny_loss.item(), "steps": 300}
del tiny_model, tiny_optimizer

tiny training mse: 1.313263 -> 0.000000


## train three seeds

settings are fixed before test evaluation. choose each seed's checkpoint by
validation rmse, with a 150-epoch cap and patience of 20. report all three seeds,
not just the best one. there is no test-based tuning.

In [7]:
runs = []
checkpoint_dir = data_dir / "rating_checkpoints"
checkpoint_dir.mkdir(exist_ok=True)
for seed in seeds:
    torch.manual_seed(seed)
    model = RatingModel(len(user_map), len(restaurant_map), training_mean).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    best_rmse, best_epoch, stale = float("inf"), 0, 0
    best_state = None
    learning_curve = []
    run_start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        predictions = model(history_edges, reverse_edges, train_indices)
        loss = nn.functional.mse_loss(predictions, train_labels)
        assert torch.isfinite(loss)
        loss.backward()
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_scores = metrics(model(history_edges, reverse_edges, val_indices), val_labels)
        learning_curve.append({"epoch": epoch, "train_mse": loss.item(), **val_scores})
        if val_scores["rmse"] < best_rmse - 1e-5:
            best_rmse, best_epoch, stale = val_scores["rmse"], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if epoch == 1 or epoch % 25 == 0:
            print(f"seed {seed} epoch {epoch}: train mse {loss.item():.4f}, val rmse {val_scores['rmse']:.4f}")
        if stale >= patience:
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        best_validation = metrics(model(history_edges, reverse_edges, val_indices), val_labels)
        train_scores = metrics(model(history_edges, reverse_edges, train_indices), train_labels)
    assert abs(best_validation["rmse"] - best_rmse) < 1e-6
    assert torch.equal(history_edges, original_history_edges)
    training_seconds = time.perf_counter() - run_start
    checkpoint_path = checkpoint_dir / f"seed_{seed}.pt"
    torch.save({"model_state": best_state, "user_map": user_map,
                "restaurant_map": restaurant_map, "training_mean": training_mean,
                "seed": seed, "best_epoch": best_epoch,
                "history_edges": history_edges}, checkpoint_path)
    runs.append({"seed": seed, "best_epoch": best_epoch, "epochs_run": epoch,
                 "training_seconds": training_seconds,
                 "seconds_per_epoch": training_seconds / epoch,
                 "train": train_scores, "validation": best_validation,
                 "learning_curve": learning_curve, "checkpoint": str(checkpoint_path)})
    print(f"seed {seed} done: best epoch {best_epoch}, {training_seconds:.2f}s, validation {best_validation}")

seed 42 epoch 1: train mse 1.9606, val rmse 1.4882


seed 42 epoch 25: train mse 1.3093, val rmse 1.3827


seed 42 done: best epoch 26, 0.58s, validation {'mae': 1.1633792716727618, 'rmse': 1.378168450655044}
seed 43 epoch 1: train mse 1.9606, val rmse 1.4882


seed 43 epoch 25: train mse 1.4922, val rmse 1.4002


seed 43 done: best epoch 28, 0.60s, validation {'mae': 1.1590927906713002, 'rmse': 1.3797624819672116}
seed 44 epoch 1: train mse 1.9606, val rmse 1.4882


seed 44 epoch 25: train mse 1.4821, val rmse 1.4042


seed 44 done: best epoch 28, 0.61s, validation {'mae': 1.1803347828975688, 'rmse': 1.3889528053460458}


## evaluate the untouched test period

all training and checkpoint selection are finished. evaluate the baseline and
every seed on exactly the same known-user, known-restaurant pairs. mae is the
average absolute error in stars; rmse gives larger mistakes more weight.

In [8]:
test_indices, test_labels = target_tensors["test"]
baseline_test = metrics(torch.full_like(test_labels, training_mean), test_labels)
for run in runs:
    checkpoint = torch.load(run["checkpoint"], map_location=device, weights_only=True)
    model = RatingModel(len(user_map), len(restaurant_map), training_mean).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    with torch.no_grad():
        predictions = model(history_edges, reverse_edges, test_indices)
    assert ((predictions >= 1) & (predictions <= 5)).all()
    run["test"] = metrics(predictions, test_labels)

model_summary = {}
for metric in ("mae", "rmse"):
    values = [run["test"][metric] for run in runs]
    model_summary[metric] = {"mean": statistics.mean(values),
                             "std": statistics.stdev(values),
                             "baseline": baseline_test[metric],
                             "relative_reduction_percent": 100 * (baseline_test[metric] - statistics.mean(values)) / baseline_test[metric]}
print("test metrics (stars; lower is better)")
print(f"training-average baseline: mae={baseline_test['mae']:.6f} rmse={baseline_test['rmse']:.6f}")
for run in runs:
    print(f"mean sage seed {run['seed']}: mae={run['test']['mae']:.6f} rmse={run['test']['rmse']:.6f}")
print("three-seed summary:", json.dumps(model_summary, indent=2))

# memory figures below cover graph tensors and model state, not total process rss
model_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
graph_bytes = sum(t.numel() * t.element_size() for t in (history_edges, reverse_edges, history_stars))
results = {
    "experiment": "boise_frozen_history_mean_sage_v1",
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "torch_geometric": torch_geometric.__version__},
    "device": str(device), "threads": torch.get_num_threads(),
    "config": {"seeds": seeds, "embedding_dim": embedding_dim, "hidden_dim": hidden_dim,
               "learning_rate": learning_rate, "weight_decay": weight_decay,
               "max_epochs": max_epochs, "patience": patience,
               "split_fractions": [0.6, 0.2, 0.1, 0.1],
               "checkpoint_selection": "minimum validation rmse per seed",
               "rating_output": "1 + 4 * sigmoid(logit)",
               "uses_history_ratings": False},
    "source_files": {str(path): {"bytes": path.stat().st_size,
                     "mtime_ns": path.stat().st_mtime_ns} for path in (business_path, review_path)},
    "sample": sample_summary, "splits": split_summary,
    "baseline": {"training_mean": training_mean, "validation": baseline_validation, "test": baseline_test},
    "runs": runs, "model_test_summary": model_summary, "tiny_training_check": tiny_check,
    "model_parameter_count": sum(p.numel() for p in model.parameters()),
    "model_parameter_mib": model_bytes / 1024**2,
    "graph_tensor_mib": graph_bytes / 1024**2,
    "total_seconds": time.perf_counter() - experiment_start,
    "checks": {"chronological": True, "target_pairs_disjoint": True,
               "reverse_edges_disjoint": True, "finite_losses_gradients_predictions": True,
               "history_graph_unchanged": True, "test_used_after_training": True},
}
results_dir = data_dir / "rating_results"
results_dir.mkdir(exist_ok=True)
(results_dir / "rating_experiment.json").write_text(json.dumps(results, indent=2, allow_nan=False) + "\n")
print(f"total runtime: {results['total_seconds']:.2f}s")
print(f"parameters: {results['model_parameter_count']:,}; parameter memory: {results['model_parameter_mib']:.2f} MiB")
print(f"graph tensor memory: {results['graph_tensor_mib']:.2f} MiB (excludes optimizer and activations)")

test metrics (stars; lower is better)
training-average baseline: mae=1.264243 rmse=1.495799
mean sage seed 42: mae=1.156392 rmse=1.385361
mean sage seed 43: mae=1.160402 rmse=1.393221
mean sage seed 44: mae=1.167578 rmse=1.388066
three-seed summary: {
  "mae": {
    "mean": 1.161457488959222,
    "std": 0.00566707138879091,
    "baseline": 1.2642429949955016,
    "relative_reduction_percent": 8.130201744692712
  },
  "rmse": {
    "mean": 1.3888826485978447,
    "std": 0.003993258108416415,
    "baseline": 1.4957991825097567,
    "relative_reduction_percent": 7.147786625509446
  }
}
total runtime: 23.68s
parameters: 286,993; parameter memory: 1.09 MiB
graph tensor memory: 1.33 MiB (excludes optimizer and activations)


## what to take away

compare the printed baseline against the three-seed average, not a cherry-picked
seed. the standard deviation is variation across initializations, not a confidence
interval for new data. rmse being higher than mae suggests uneven error sizes,
but does not identify their cause.

this only evaluates pairs whose users and restaurants existed in the history graph.
the graph stays fixed even as time passes. history ratings aren't used, and beating
a constant baseline wouldn't establish that graph aggregation itself caused the gain.
a stronger baseline and a no-message-passing comparison would be later work.

see `milestone_report.md` for the actual results and interpretation.
api references: [SAGEConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.SAGEConv.html),
[Embedding](https://docs.pytorch.org/docs/2.15/generated/torch.nn.Embedding.html),
[MSELoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.MSELoss.html).

actual result interpretation:

On the 794 eligible test reviews, the mean-aggregation model had an average MAE of 1.161 stars and RMSE of 1.389 stars across three random seeds. The training-average baseline had MAE 1.264 and RMSE 1.496, so the model reduced these errors by 8.1% and 7.1%, respectively. The MAE means predictions were off by about 1.16 stars on average; the higher RMSE shows that larger mistakes still matter. These results give initial evidence that the trained predictor can outperform a single average rating on this restricted future-review task, but the remaining error is substantial. They do not establish that message passing caused the improvement, and the test only covers 12.3% of the later review pairs because both nodes had to exist in the frozen history graph.